# Taller 2 (GCP): ETL con Apache Spark y Delta Lake en Dataproc

Implementarás el mismo flujo ETL Medallion (Bronze → Silver → Gold) del Taller Databricks,  
pero sobre **Google Cloud Platform** usando **Dataproc** (Spark administrado) y **Cloud Storage (GCS)** como data lake.

## Diferencias clave respecto a Databricks
| Concepto | Databricks | GCP Dataproc |
|---|---|---|
| Almacenamiento | DBFS (`/tmp/...`) | Cloud Storage (`gs://bucket/...`) |
| Operaciones de archivos | `dbutils.fs` | `gsutil` (CLI) o `google-cloud-storage` |
| Visualización en notebook | `display(df)` | `df.show()` / `df.limit(n).toPandas()` |
| Delta Lake | Pre-instalado | Configurado al crear el cluster |
| SparkSession | Pre-inicializada | Pre-inicializada (kernel PySpark) |

---
## 0. Configuración del entorno en GCP

Sigue estos pasos **una sola vez** antes de ejecutar el notebook.

---

### Paso 1 — Crear una cuenta en Google Cloud Platform

GCP ofrece **$300 USD en créditos gratuitos** por 90 días. No se cobra nada automáticamente al terminar el periodo.

**Windows / macOS / Linux:**
1. Ve a https://cloud.google.com/free
2. Haz clic en **"Comenzar gratis"** / **"Get started for free"**
3. Inicia sesión con una cuenta de Google
4. Completa el formulario (requiere tarjeta de crédito solo para verificación de identidad)
5. Activa la prueba gratuita y confirma que aparecen los $300 de crédito en la consola

> Si ya tienes cuenta de GCP con créditos activos, ve directamente al Paso 2.

---

### Paso 2 — Crear un proyecto

Todos los recursos de GCP viven dentro de un proyecto.

1. Ve a https://console.cloud.google.com
2. En la barra superior, haz clic en el **selector de proyectos** (junto al logo de Google Cloud)
3. Haz clic en **"Nuevo proyecto"**
4. **Nombre del proyecto:** `taller2-spark`
5. Haz clic en **"Crear"**
6. Asegúrate de tener el nuevo proyecto seleccionado en el selector

---

### Paso 3 — Habilitar las APIs necesarias

Ve a cada enlace con tu proyecto seleccionado y haz clic en **"Habilitar"**:
- **Cloud Dataproc API:** https://console.cloud.google.com/apis/library/dataproc.googleapis.com
- **Cloud Storage API:** https://console.cloud.google.com/apis/library/storage.googleapis.com

---

### Paso 4 — Crear un bucket en Cloud Storage (GCS)

GCS es el equivalente al DBFS de Databricks: el sistema de almacenamiento distribuido del data lake.

1. Ve a https://console.cloud.google.com/storage/browser
2. Haz clic en **"Crear bucket"**
3. **Nombre del bucket:** elige un nombre único globalmente, por ejemplo `taller2-spark-tunombre`  
   *(los nombres de bucket son globales en GCP, no puede repetirse entre usuarios)*
4. **Región:** selecciona la más cercana a tu ubicación (recomendado: `us-central1` o `southamerica-east1` para Colombia)
5. **Clase de almacenamiento:** Standard
6. Deja el resto por defecto y haz clic en **"Crear"**

> Anota el nombre exacto del bucket. Lo usarás en la variable `BUCKET_NAME` del notebook.

---

### Paso 5 — Crear el cluster de Dataproc con Delta Lake

Dataproc es el servicio de Spark administrado de GCP. El cluster incluye JupyterLab integrado.

1. Ve a https://console.cloud.google.com/dataproc/clusters
2. Haz clic en **"Crear cluster"** → **"Cluster en Compute Engine"**
3. Configura la sección **"Configuración del cluster"**:
   - **Nombre:** `taller2-cluster`
   - **Región:** la misma que elegiste para el bucket
   - **Tipo de cluster:** `Nodo único` (Single Node) — suficiente para el taller y más económico
   - **Versión de imagen:** elige la versión más reciente con etiqueta `LTS` (ej. `2.1-debian11`)
4. Despliega **"Componentes opcionales"** y habilita: **Jupyter Notebook**
5. Despliega **"Personalización del cluster"** → **"Propiedades"** y agrega estas tres propiedades para activar Delta Lake:

| Prefijo | Clave | Valor |
|---|---|---|
| `spark` | `spark.jars.packages` | `io.delta:delta-core_2.12:2.4.0` |
| `spark` | `spark.sql.extensions` | `io.delta.sql.DeltaSparkSessionExtension` |
| `spark` | `spark.sql.catalog.spark_catalog` | `org.apache.spark.sql.delta.catalog.DeltaCatalog` |

6. Haz clic en **"Crear cluster"** y espera 3-5 minutos hasta que el estado sea **En ejecución** (círculo verde)

> **Costo estimado:** un cluster Single Node con la configuración por defecto consume aproximadamente $0.10-0.20 USD/hora de tus créditos. Recuerda **eliminarlo** cuando termines el taller.

---

### Paso 6 — Abrir JupyterLab en Dataproc

1. En la lista de clusters, haz clic sobre `taller2-cluster`
2. Ve a la pestaña **"Interfaces web"**
3. Haz clic en **"JupyterLab"** — se abrirá en una nueva pestaña
4. En JupyterLab, ve a **File → Upload Files** y sube el archivo `guia_gcp_dataproc.ipynb`
5. Abre el notebook y **verifica que el kernel sea PySpark** (esquina superior derecha)
   - Si dice `Python 3`, haz clic → selecciona **PySpark**

> En Dataproc con kernel PySpark, `spark` y `sc` (SparkContext) están disponibles automáticamente. No necesitas crear SparkSession.

---

### Paso 7 — Actualizar el nombre del bucket en el notebook

En la primera celda de código, reemplaza `TU_BUCKET` con el nombre exacto del bucket que creaste.

---
## Arquitectura del taller

```
  DATOS SINTETICOS
  (generados con Spark)
          |
          v
 +----------------+
 |    BRONZE      |  gs://bucket/taller2/bronze/
 |   Delta Lake   |  Datos crudos, sin transformar
 +----------------+
          |
          v
 +----------------+
 |    SILVER      |  gs://bucket/taller2/silver/
 |   Delta Lake   |  Datos limpios y enriquecidos
 +----------------+
          |
          v
 +----------------+
 |     GOLD       |  gs://bucket/taller2/gold/
 |   Delta Lake   |  Metricas para consumo analitico
 +----------------+
```

**Almacenamiento:** Google Cloud Storage (GCS)  
**Formato:** Delta Lake (Parquet + transaction log)

In [ ]:
# =============================================================
# CONFIGURACIÓN — ACTUALIZA ESTE VALOR ANTES DE CONTINUAR
# =============================================================

BUCKET_NAME = "TU_BUCKET"  # <-- reemplaza con el nombre de tu bucket GCS

# Rutas en Cloud Storage (equivalentes al DBFS de Databricks)
BASE_PATH   = f"gs://{BUCKET_NAME}/taller2"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH   = f"{BASE_PATH}/gold"

# Limpiar ejecuciones anteriores (gsutil reemplaza a dbutils.fs.rm)
import subprocess
subprocess.run(["gsutil", "-m", "rm", "-rf", f"gs://{BUCKET_NAME}/taller2/"],
               capture_output=True)
print("Directorio anterior eliminado (si existia).")

print(f"\nRutas configuradas:")
print(f"  Bronze : {BRONZE_PATH}")
print(f"  Silver : {SILVER_PATH}")
print(f"  Gold   : {GOLD_PATH}")
print(f"\nSpark version: {spark.version}")

---
## Parte 1 — Bronze: Ingesta de datos crudos

Generamos 1.000 transacciones de ventas sintéticas con problemas de calidad intencionales  
y las guardamos como **Delta Lake en GCS** (equivalente al DBFS de Databricks).

In [ ]:
# =============================================================
# 1.1 — GENERAR DATOS SINTETICOS
# =============================================================

from pyspark.sql import functions as F

SEED = 42

arr_categorias = F.array([F.lit(c) for c in ["Electronica", "Muebles", "Ropa", "Alimentos", "Libros"]])
arr_ciudades   = F.array([F.lit(c) for c in ["Bogota", "Medellin", "Cali", "Barranquilla", "Cartagena"]])
arr_pagos      = F.array([F.lit(p) for p in ["tarjeta_credito", "tarjeta_debito", "efectivo", "transferencia"]])
arr_productos  = F.array([F.lit(p) for p in [
    "Laptop Pro", "Silla Ergonomica", "Camiseta Algodon", "Arroz Premium",
    "Python Avanzado", "Monitor 4K", "Escritorio Pie", "Jean Clasico",
    "Aceite Oliva", "SQL para Todos", "Auriculares BT", "Estante Madera",
    "Zapatos Cuero", "Leche Entera", "Historia IA"
]])

df_bronze_raw = (
    spark.range(1, 1001)
    .select(
        F.col("id"),
        F.rand(SEED).alias("r1"),
        F.rand(SEED + 1).alias("r2"),
        F.rand(SEED + 2).alias("r3"),
        F.rand(SEED + 3).alias("r4"),
        F.rand(SEED + 4).alias("r5"),
        F.rand(SEED + 5).alias("r6"),
        F.rand(SEED + 6).alias("r7"),
    )
    .select(
        F.concat(F.lit("TXN-"), F.lpad(F.col("id").cast("string"), 5, "0")).alias("transaction_id"),
        F.concat(F.lit("C"), F.lpad(((F.col("r1") * 200).cast("int") + 1).cast("string"), 4, "0")).alias("customer_id"),
        arr_productos[(F.col("r2") * 15).cast("int")].alias("product_name"),
        arr_categorias[(F.col("r3") * 5).cast("int")].alias("category"),
        ((F.col("r4") * 9).cast("int") + 1).alias("quantity"),
        F.when(F.col("r5") < 0.03, F.lit(-99.0))
         .otherwise(F.round(F.col("r5") * 1995 + 5, 2)).alias("unit_price"),
        F.date_add(F.lit("2024-01-01").cast("date"), (F.col("r6") * 365).cast("int")).alias("transaction_date"),
        arr_ciudades[(F.col("r1") * 5).cast("int")].alias("store_city"),
        arr_pagos[(F.col("r2") * 4).cast("int")].alias("payment_method"),
        F.when(F.col("r7") > 0.9, F.lit(None).cast("double"))
         .otherwise((F.col("r4") * 4).cast("int") * 0.05).alias("discount_pct"),
    )
)

print(f"Registros generados: {df_bronze_raw.count():,}")
df_bronze_raw.printSchema()
df_bronze_raw.show(5, truncate=False)

In [ ]:
# =============================================================
# 1.2 — GUARDAR BRONZE EN GCS COMO DELTA LAKE
# Delta Lake escribe en GCS exactamente igual que en DBFS.
# La diferencia es solo el prefijo de la ruta: gs:// vs /tmp/
# =============================================================

(
    df_bronze_raw
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{BRONZE_PATH}/ventas/")
)

print(f"Tabla Bronze guardada en: {BRONZE_PATH}/ventas/")

# Ver archivos en GCS (gsutil ls reemplaza a dbutils.fs.ls)
result = subprocess.run(
    ["gsutil", "ls", f"{BRONZE_PATH}/ventas/"],
    capture_output=True, text=True
)
print("\nArchivos en Bronze (GCS):")
print(result.stdout)

In [ ]:
# =============================================================
# 1.3 — EXPLORAR BRONZE
# Nota: en Dataproc usamos .show() en lugar de display()
# =============================================================

df_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/ventas/")

print(f"Total registros  : {df_bronze.count():,}")
print(f"Total columnas   : {len(df_bronze.columns)}")

print("\n=== Nulos por columna ===")
df_bronze.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_bronze.columns
]).show()

print(f"Precios invalidos (< 0): {df_bronze.filter(F.col('unit_price') < 0).count()}")

print("\n=== Distribucion por categoria ===")
df_bronze.groupBy("category").count().orderBy("count", ascending=False).show()

# Para visualizacion mas rica en Jupyter, convertir a Pandas (datasets pequenos)
print("\n=== Muestra (primeras 5 filas) ===")
df_bronze.limit(5).toPandas()

---
## Parte 2 — Silver: Limpieza y transformación

In [ ]:
# =============================================================
# 2.1 — TRANSFORMACION SILVER
# Leer Bronze desde GCS → limpiar → enriquecer → guardar Silver en GCS
# =============================================================

df_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/ventas/")

df_silver = (
    df_bronze
    .filter(F.col("unit_price") > 0)
    .fillna({"discount_pct": 0.0})
    .withColumn("total_bruto",    F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("total_descuento",F.round(F.col("total_bruto") * F.col("discount_pct"), 2))
    .withColumn("total_neto",     F.round(F.col("total_bruto") - F.col("total_descuento"), 2))
    .withColumn("year",           F.year("transaction_date"))
    .withColumn("month",          F.month("transaction_date"))
    .withColumn("month_name",     F.date_format("transaction_date", "MMMM"))
    .withColumn("day_of_week",    F.date_format("transaction_date", "EEEE"))
    .withColumn("processed_at",   F.current_timestamp())
)

print(f"Bronze  : {df_bronze.count():,} registros")
print(f"Silver  : {df_silver.count():,} registros")
print(f"Eliminados: {df_bronze.count() - df_silver.count():,}")

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("year", "month")
    .save(f"{SILVER_PATH}/ventas/")
)

print(f"\nSilver guardada en: {SILVER_PATH}/ventas/")

In [ ]:
# =============================================================
# 2.2 — VERIFICAR SILVER
# =============================================================

df_silver_check = spark.read.format("delta").load(f"{SILVER_PATH}/ventas/")

print("=== Nulos en columnas criticas (deben ser 0) ===")
df_silver_check.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["unit_price", "discount_pct", "total_neto"]
]).show()

print("=== Muestra de datos Silver ===")
df_silver_check.select(
    "transaction_id", "customer_id", "category",
    "quantity", "unit_price", "discount_pct", "total_neto",
    "year", "month"
).show(10)

---
## Parte 3 — Gold: Métricas y agregaciones

In [ ]:
# =============================================================
# 3.1 — VENTAS POR CATEGORIA Y MES
# =============================================================

df_silver = spark.read.format("delta").load(f"{SILVER_PATH}/ventas/")

gold_cat_mes = (
    df_silver
    .groupBy("year", "month", "month_name", "category")
    .agg(
        F.count("transaction_id").alias("num_transacciones"),
        F.sum("quantity").alias("unidades_vendidas"),
        F.round(F.sum("total_bruto"), 2).alias("ingresos_brutos"),
        F.round(F.sum("total_neto"), 2).alias("ingresos_netos"),
        F.round(F.avg("unit_price"), 2).alias("precio_promedio"),
    )
    .orderBy("year", "month", F.col("ingresos_netos").desc())
)

gold_cat_mes.show(15)
gold_cat_mes.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/ventas_categoria_mes/")
print(f"Guardado en: {GOLD_PATH}/ventas_categoria_mes/")

In [ ]:
# =============================================================
# 3.2 — TOP CLIENTES + SEGMENTACION
# =============================================================

gold_clientes = (
    df_silver
    .groupBy("customer_id")
    .agg(
        F.count("transaction_id").alias("num_compras"),
        F.round(F.sum("total_neto"), 2).alias("gasto_total"),
        F.round(F.avg("total_neto"), 2).alias("gasto_promedio"),
        F.countDistinct("category").alias("categorias_distintas"),
    )
    .withColumn(
        "segmento",
        F.when(F.col("gasto_total") >= 5000, "VIP")
         .when(F.col("gasto_total") >= 2000, "Premium")
         .when(F.col("gasto_total") >= 500,  "Regular")
         .otherwise("Ocasional")
    )
    .orderBy(F.col("gasto_total").desc())
)

print("=== Top 15 clientes ===")
gold_clientes.show(15)

print("=== Distribucion de segmentos ===")
gold_clientes.groupBy("segmento").count().orderBy("count", ascending=False).show()

gold_clientes.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/top_clientes/")
print(f"Guardado en: {GOLD_PATH}/top_clientes/")

In [ ]:
# =============================================================
# 3.3 — ANALISIS POR METODO DE PAGO (Spark SQL)
# =============================================================

df_silver.createOrReplaceTempView("silver_ventas")

gold_pagos = spark.sql("""
    SELECT
        payment_method,
        COUNT(*)                           AS num_transacciones,
        ROUND(SUM(total_neto), 2)          AS ingresos_netos,
        ROUND(AVG(total_neto), 2)          AS ticket_promedio,
        ROUND(AVG(discount_pct) * 100, 1)  AS descuento_promedio_pct
    FROM silver_ventas
    GROUP BY payment_method
    ORDER BY ingresos_netos DESC
""")

gold_pagos.show()
gold_pagos.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/metodos_pago/")
print(f"Guardado en: {GOLD_PATH}/metodos_pago/")

---
## Parte 4 — Delta Lake: Capacidades avanzadas

Delta Lake funciona de forma idéntica sobre GCS que sobre DBFS.  
El transaction log (`_delta_log/`) se almacena directamente en el bucket de GCS.

In [ ]:
# =============================================================
# 4.1 — TRANSACTION LOG: DESCRIBE HISTORY
# =============================================================

from delta.tables import DeltaTable

silver_dt = DeltaTable.forPath(spark, f"{SILVER_PATH}/ventas/")

print("=== Historial de la tabla Silver ===")
silver_dt.history().select(
    "version", "timestamp", "operation", "numOutputRows"
).show(truncate=False)

# Ver el transaction log en GCS
result = subprocess.run(
    ["gsutil", "ls", f"{SILVER_PATH}/ventas/_delta_log/"],
    capture_output=True, text=True
)
print("\nArchivos del transaction log en GCS:")
print(result.stdout)

In [ ]:
# =============================================================
# 4.2 — CREAR NUEVA VERSION CON APPEND
# =============================================================

df_nuevos = df_silver.limit(50).withColumn("processed_at", F.current_timestamp())

(
    df_nuevos
    .write
    .format("delta")
    .mode("append")
    .save(f"{SILVER_PATH}/ventas/")
)

print("Append completado. Versiones disponibles:")
silver_dt.history().select("version", "timestamp", "operation").show()

In [ ]:
# =============================================================
# 4.3 — TIME TRAVEL: LEER VERSION ANTERIOR
# Funciona igual en GCS que en DBFS gracias al transaction log
# =============================================================

df_actual = spark.read.format("delta").load(f"{SILVER_PATH}/ventas/")

df_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(f"{SILVER_PATH}/ventas/")
)

print(f"Version actual  : {df_actual.count():,} registros")
print(f"Version 0       : {df_v0.count():,} registros")
print(f"Diferencia      : {df_actual.count() - df_v0.count():,} registros")

In [ ]:
# =============================================================
# 4.4 — OPTIMIZE: COMPACTAR ARCHIVOS EN GCS
# =============================================================

spark.sql(f"OPTIMIZE delta.`{SILVER_PATH}/ventas/`")

print("OPTIMIZE completado.")
silver_dt.history().select("version", "operation").show(5)

---
## Parte 5 — Ejercicios integradores

Los mismos ejercicios del Taller Databricks, ahora ejecutándose sobre GCP.

**E1** — Producto más vendido (en unidades) por categoría.

In [ ]:
# Tu respuesta aquí


**E2** — Ingresos netos y tasa de descuento promedio por ciudad (`store_city`). Ordena de mayor a menor ingreso.

In [ ]:
# Tu respuesta aquí


**E3** — Tabla Gold `tendencia_semanal` con: semana del año, número de transacciones, ingresos netos y ticket promedio. Guárdala en `{GOLD_PATH}/tendencia_semanal/` como Delta.

In [ ]:
# Tu respuesta aquí


---
## Soluciones de referencia

In [ ]:
# --- E1 ---
# from pyspark.sql import Window
# w = Window.partitionBy("category").orderBy(F.col("total_unidades").desc())
# (
#     df_silver.groupBy("category", "product_name")
#     .agg(F.sum("quantity").alias("total_unidades"))
#     .withColumn("rank", F.rank().over(w))
#     .filter(F.col("rank") == 1).drop("rank")
#     .orderBy("category").show()
# )

# --- E2 ---
# (
#     df_silver.groupBy("store_city")
#     .agg(
#         F.round(F.avg("discount_pct") * 100, 1).alias("descuento_prom_pct"),
#         F.round(F.sum("total_neto"), 2).alias("ingresos_netos")
#     )
#     .orderBy(F.col("ingresos_netos").desc()).show()
# )

# --- E3 ---
# gold_semanal = (
#     df_silver
#     .withColumn("week", F.weekofyear("transaction_date"))
#     .groupBy("year", "week")
#     .agg(
#         F.count("transaction_id").alias("num_transacciones"),
#         F.round(F.sum("total_neto"), 2).alias("ingresos_netos"),
#         F.round(F.avg("total_neto"), 2).alias("ticket_promedio")
#     )
#     .orderBy("year", "week")
# )
# gold_semanal.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/tendencia_semanal/")
# gold_semanal.show()

---
## Limpieza de recursos

**Importante:** para evitar consumo innecesario de créditos, elimina el cluster y los datos cuando termines.

In [ ]:
# Eliminar datos del taller del bucket GCS
result = subprocess.run(
    ["gsutil", "-m", "rm", "-rf", f"gs://{BUCKET_NAME}/taller2/"],
    capture_output=True, text=True
)
print(f"Datos eliminados de gs://{BUCKET_NAME}/taller2/")

### Eliminar el cluster de Dataproc

1. Ve a https://console.cloud.google.com/dataproc/clusters
2. Selecciona `taller2-cluster`
3. Haz clic en **"Eliminar"** y confirma

> Si no eliminas el cluster, seguirá consumiendo créditos aunque no lo estés usando.